# Probing Liquid Heights

- tags: #liquidhandling #hamiltonstar #lld #probing
- Last updated: 2026-02-28

`probe_liquid_heights` measures the liquid surface height in one or more containers
using capacitive (cLLD) or pressure-based (pLLD) liquid level detection. It handles
multi-channel batching automatically — grouping containers by X position, partitioning
channels into parallel-compatible Y batches, and interpolating phantom channel positions
for non-consecutive channel use.

This notebook walks through progressively complex scenarios:

1. Single well, single channel
2. Multiple wells across a plate
3. Wide trough with multiple channels
4. Containers at different X positions (multi-batch)
5. Inspecting the execution plan with `get_probing_plan()`

## Prerequisites

- Machines used:
  - Hamilton STAR
- Non-PLR dependencies: None

## Protocol Mode

In [1]:
protocol_mode = "simulation"  # "execution" or "simulation"

---
## Import Statements

### Machine & Visualizer

In [2]:
%load_ext autoreload
%autoreload 2

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends.hamilton.STAR_backend import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck
from pylabrobot.visualizer.visualizer import Visualizer

if protocol_mode == "execution":
    star_backend = STARBackend()

elif protocol_mode == "simulation":
    from pylabrobot.liquid_handling.backends.hamilton.STAR_chatterbox import STARChatterboxBackend

    star_backend = STARChatterboxBackend()

### Required Resources

In [3]:
from pylabrobot.resources import (
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    Trough_CAR_4R200_A00,
    hamilton_96_tiprack_1000uL,
    Cor_96_wellplate_360ul_Fb,
    Greiner_384_wellplate_28ul_Fb,
    Porvair_24_wellplate_Vb,
    hamilton_1_trough_200ml_Vb,
)

## Instantiate Frontend & Connect to Machine

In [4]:
deck = STARLetDeck()
lh = LiquidHandler(backend=star_backend, deck=deck, name="STARlet")

await lh.setup()

vis = Visualizer(resource=lh)
await vis.setup()

Websocket server started at http://127.0.0.1:2124
File server started at http://127.0.0.1:1340 . Open this URL in your browser.


## Configure Deck Layout

In [5]:
# Tips
tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_rack = hamilton_96_tiprack_1000uL(name="tip_rack_1000")
tip_carrier[0] = tip_rack
deck.assign_child_resource(tip_carrier, rails=1)

# Plates
plate_carrier = PLT_CAR_L5AC_A00(name="plate_carrier")
plate_carrier[2] = demo_96_wellplate = Cor_96_wellplate_360ul_Fb(name="demo_96_wellplate")
plate_carrier[1] = demo_384_wellplate = Greiner_384_wellplate_28ul_Fb(name="demo_384_wellplate")
plate_carrier[0] = demo_24_wellplate = Porvair_24_wellplate_Vb(name="demo_24_wellplate")
deck.assign_child_resource(plate_carrier, rails=8)

# Trough
trough_carrier = Trough_CAR_4R200_A00(name="trough_carrier")
trough = hamilton_1_trough_200ml_Vb(name="trough_200ml")
trough_carrier[0] = trough
deck.assign_child_resource(trough_carrier, rails=15)

## Simulate Liquid Volumes (simulation only)

In simulation mode the chatterbox computes liquid heights from tracked volumes.
We pre-fill the plate with a gradient so probing returns non-zero heights.

In [6]:
if protocol_mode == "simulation":
    # 96-well plate: 8 rows x 12 cols gradient
    for ri, r in enumerate("ABCDEFGH"):
        for ci in range(1, 13):
            vol = round(20 + 25 * (ci - 1 + ri / 8), 1)
            demo_96_wellplate[f"{r}{ci}"][0].tracker.set_volume(vol)

    # 24-well plate: 2 rows x 12 cols gradient (max capacity ~1550 uL)
    for ri, r in enumerate("AB"):
        for ci in range(1, 13):
            vol = round(200 + 100 * (ci - 1 + ri / 2), 1)
            demo_24_wellplate[f"{r}{ci}"][0].tracker.set_volume(vol)

    # Trough
    trough.tracker.set_volume(150_000)  # 150 mL

    print(f"96-well volumes set (20.0 - 316.9 uL)")
    print(f"24-well volumes set (200.0 - 1350.0 uL)")
    print(f"Trough: {trough.tracker.get_used_volume()} uL")

96-well volumes set (20.0 - 316.9 uL)
24-well volumes set (200.0 - 1350.0 uL)
Trough: 150000 uL


## Pick Up Tips

All probing requires tips on the channels being used.

```{note}
Rule of thumb:
The shorter the tips the straighter the tip.
Not all tip pickups will be perfect. Some might pick up with a slight tilt which deviates the believed tip bottom x-y position from the true x-y position.
The longer the tip the stronger that deviation can become.
As a result, we recommend using the teaching needles for high-precision probing.

However, the teaching needles are not as tolerant to user mistakes (e.g. forgetting a resource is in the movement path).
As a result, we recomment Hamilton's 50 uL tips to be used for most probing actions.
```

In [7]:
await lh.pick_up_tips(tip_rack["A1:H1"])

C0TTid0001tt01tf0tl0871tv12500tg3tu0
C0TPid0002xp01179 01179 01179 01179 01179 01179 01179 01179yp1458 1368 1278 1188 1098 1008 0918 0828tm1 1 1 1 1 1 1 1tt01tp2266tz2166th2450td0


---
## 1. Single Well, Single Channel

The simplest case: probe one well with one channel. Returns the liquid height
in mm from cavity bottom.

In [8]:
heights = await star_backend.probe_liquid_heights(
    containers=demo_96_wellplate["A1"],
    use_channels=[0],
)
print(f"A1 liquid height: {heights[0]:.2f} mm")

probe_liquid_heights: ['0.54'] mm
A1 liquid height: 0.54 mm


## 2. Multiple Wells Across a Plate

Probe a column of wells using 8 channels in parallel. All wells share the same
X position, so this runs as a single batch.

In [9]:
wells = demo_96_wellplate["A1:H1"]

heights = await star_backend.probe_liquid_heights(
    containers=wells,
    use_channels=[0, 1, 2, 3, 4, 5, 6, 7],
)

for well, h in zip(wells, heights):
    print(f"{well.name}: {h:.2f} mm")

probe_liquid_heights: ['0.54', '0.62', '0.71', '0.80', '0.88', '0.96', '1.05', '1.13'] mm
demo_96_wellplate_well_A1: 0.54 mm
demo_96_wellplate_well_B1: 0.62 mm
demo_96_wellplate_well_C1: 0.71 mm
demo_96_wellplate_well_D1: 0.80 mm
demo_96_wellplate_well_E1: 0.88 mm
demo_96_wellplate_well_F1: 0.96 mm
demo_96_wellplate_well_G1: 1.05 mm
demo_96_wellplate_well_H1: 1.13 mm


## 3. Wide Trough With Multiple Channels

When multiple channels target the same container, `probe_liquid_heights`
automatically spreads them in Y to avoid collisions. The spread offsets are
computed by `compute_single_container_offsets` using the container's physical
width and the channel minimum spacing.

In [10]:
heights = await star_backend.probe_liquid_heights(
    containers=[trough] * 4,
    use_channels=[0, 1, 2, 3],
)

print(f"Trough liquid height (4-channel): {heights}")

probe_liquid_heights: ['47.80', '47.80', '47.80', '47.80'] mm
Trough liquid height (4-channel): [47.8, 47.8, 47.8, 47.8]


## 4. Non-Consecutive Channels

You don't have to use consecutive channels. When channels are non-consecutive
(e.g. `[0, 2, 5]`), phantom intermediate channels are automatically positioned
to satisfy spacing constraints.

In [11]:
heights = await star_backend.probe_liquid_heights(
    containers=demo_96_wellplate["A3", "C3", "F3"],
    use_channels=[0, 2, 5],
)

for ch, h in zip([0, 2, 5], heights):
    print(f"Channel {ch}: {h:.2f} mm")

probe_liquid_heights: ['1.89', '2.06', '2.32'] mm
Channel 0: 1.89 mm
Channel 2: 2.06 mm
Channel 5: 2.32 mm


## 5. Containers at Different X Positions (Multi-Batch)

The Hamilton STAR has a single X carriage, so containers at different X
positions must be probed in separate batches (sequentially). The method
handles this grouping automatically.

In [12]:
# Mix containers from the 96-well plate and trough — different X positions
heights = await star_backend.probe_liquid_heights(
    containers=demo_96_wellplate["A1", "B1"] + [trough],
    use_channels=[0, 1, 2],
)

print(f"96-well A1: {heights[0]:.2f} mm")
print(f"96-well B1: {heights[1]:.2f} mm")
print(f"Trough:     {heights[2]:.2f} mm")

probe_liquid_heights: ['0.54', '0.62', '47.80'] mm
96-well A1: 0.54 mm
96-well B1: 0.62 mm
Trough:     47.80 mm


---
## 6. Inspecting the Execution Plan

Use `get_probing_plan()` on the chatterbox backend to visualize how channels
will be batched **without** running any hardware. This is invaluable for
debugging and understanding the batch structure.

The plan shows:
- **X Groups**: containers sharing an X position (processed sequentially)
- **Y Batches**: channel subsets within an X group (processed sequentially)
- **Channels per batch**: which channels probe in parallel

In [13]:
if protocol_mode == "simulation":
    plan = star_backend.get_probing_plan(
        containers=demo_96_wellplate["A1", "B1", "C1"] + [trough, trough],
        use_channels=[0, 1, 2, 3, 4],
    )

    print(f"Total X groups:  {len(plan.x_groups)}")
    print(f"Total batches:   {plan.total_batches}")
    print(f"Fully parallel:  {plan.is_fully_parallel}")
    for i, xg in enumerate(plan.x_groups):
        print(f"\n  X Group {i}: x = {xg.x_position:.1f} mm")
        for j, yb in enumerate(xg.y_batches):
            print(f"    Y Batch {j}: channels {yb.channels}, containers {yb.container_names}")
            if yb.intermediate_y_positions:
                print(f"      Phantom channels: {yb.intermediate_y_positions}")

Total X groups:  2
Total batches:   3
Fully parallel:  False

  X Group 0: x = 275.8 mm
    Y Batch 0: channels [0, 1, 2], containers ['demo_96_wellplate_well_A1', 'demo_96_wellplate_well_B1', 'demo_96_wellplate_well_C1']

  X Group 1: x = 437.5 mm
    Y Batch 0: channels [3], containers ['trough_200ml']
    Y Batch 1: channels [4], containers ['trough_200ml']


### Y Sub-Batching: 384-Well Plate

The 384-well plate has 4.5mm well-to-well pitch, but the minimum channel spacing
is 9mm. This means adjacent rows (e.g. A1 and B1) **cannot be probed in parallel** —
the orchestrator automatically splits them into sequential Y batches within the
same X group.

In [14]:
if protocol_mode == "simulation":
    # Probe 4 adjacent 384-well rows at once — channels can't all fit in one Y batch
    wells_384 = demo_384_wellplate["A1", "B1", "C1", "D1"]

    plan = star_backend.get_probing_plan(
        containers=wells_384,
        use_channels=[0, 1, 2, 3],
    )

    print(f"384-well: 4 adjacent rows (4.5 mm pitch, 9 mm min channel spacing)")
    print(f"Total Y batches: {plan.total_batches}")
    print(f"Fully parallel:  {plan.is_fully_parallel}\n")
    for i, xg in enumerate(plan.x_groups):
        print(f"  X Group {i}: x = {xg.x_position:.1f} mm")
        for j, yb in enumerate(xg.y_batches):
            print(f"    Y Batch {j}: channels {yb.channels}, containers {yb.container_names}")

384-well: 4 adjacent rows (4.5 mm pitch, 9 mm min channel spacing)
Total Y batches: 4
Fully parallel:  False

  X Group 0: x = 273.4 mm
    Y Batch 0: channels [0], containers ['demo_384_wellplate_well_A1']
    Y Batch 1: channels [1], containers ['demo_384_wellplate_well_B1']
    Y Batch 2: channels [2], containers ['demo_384_wellplate_well_C1']
    Y Batch 3: channels [3], containers ['demo_384_wellplate_well_D1']


## 7. LLD Mode Selection

The default LLD mode is `GAMMA` (capacitive). You can switch to pressure-based LLD:

| Mode | Description |
|------|-------------|
| `LLDMode.GAMMA` | Capacitive LLD (cLLD) — default, fast, reliable for aqueous/conductive liquids |
| `LLDMode.PRESSURE` | Pressure-based LLD (pLLD) — for non-conductive liquids (organic solvents, oils) where cLLD fails; also has foam detection parameters |

```{note}
Many automators need to see the liquid move in/out of the tip.
This is not possible with (most) cLLD-compatible (most commonly black) tips.
Using pLLD enables the usage of transparent (and therefore highly charge-resistant) tips.
```

In [15]:
# Capacitive LLD (default)
heights_clld = await star_backend.probe_liquid_heights(
    containers=demo_96_wellplate["A2"],
    use_channels=[0],
    lld_mode=STARBackend.LLDMode.GAMMA,
)

# Pressure-based LLD
heights_plld = await star_backend.probe_liquid_heights(
    containers=demo_96_wellplate["A3"],
    use_channels=[0],
    lld_mode=STARBackend.LLDMode.PRESSURE,
)

print(f"cLLD: {heights_clld[0]:.2f} mm")
print(f"pLLD: {heights_plld[0]:.2f} mm")

probe_liquid_heights: ['1.22'] mm
probe_liquid_heights: ['1.89'] mm
cLLD: 1.22 mm
pLLD: 1.89 mm


## 8. Useful Parameters

Key parameters you may want to tune:

- **`search_speed`** (default 10.0 mm/s): How fast the channel descends during detection.
  Lower = more accurate, higher = faster.
- **`n_replicates`** (default 1): Number of repeated measurements. Results are averaged.
- **`move_to_z_safety_after`** (default True): Whether to move channels to safe Z height
  after probing. Set to `False` if you want to chain operations without full Z retraction.

In [16]:
# Slower, more accurate probing with 3 replicates
heights = await star_backend.probe_liquid_heights(
    containers=demo_24_wellplate["A1"],
    use_channels=[0],
    search_speed=5.0,
    n_replicates=3,
)
print(f"24-well A1 height (3x averaged): {heights[0]:.2f} mm")

probe_liquid_heights: ['6.63'] mm
24-well A1 height (3x averaged): 6.63 mm


---
## Clean Up

In [17]:
await lh.drop_tips(tip_rack["A1:H1"])

C0TRid0003xp01179 01179 01179 01179 01179 01179 01179 01179yp1458 1368 1278 1188 1098 1008 0918 0828tm1 1 1 1 1 1 1 1tp2266tz2186th2450te2450ti1


In [18]:
await vis.stop()
await lh.stop()